# 01 · The baseline run

*CRNTiny, 25 epochs, stock hyperparameters*

Neural Audio and Speech Processing — Day 2 / Application to Speech Enhancement
Homework assignment, R. Scheibler (2026-07-02) · dataset: Voicebank-DEMAND (16 kHz)

This notebook produces the run that every later experiment is compared against,
and the `runs/<baseline>` argument of the final `create_report.py` call.

The configuration is byte-for-byte equivalent to `config/default_config.py`
(`CRNTiny`, 25 epochs, batch 32, `lr=1e-3`, `wd=0.02`, seed 42, no remix); it is
written out under a different `name` only so the run folder is easy to spot.

**Runtime:** ~45–70 min on a T4.

In [ ]:
# --- Google Colab bootstrap (does nothing when running locally) --------------
# IMPORTANT: point this at the fork that contains the homework modifications
# (models/crn.py with a selectable activation, train.py with --warmup-steps).
REPO_URL = "https://github.com/Ahmed-AlGhosaini/nanoSE.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os

    if not os.path.exists("nanoSE"):
        !git clone -q $REPO_URL nanoSE
    %cd nanoSE
    !pip install -q -r requirements.txt

    import torch
    if not torch.cuda.is_available():
        print("No GPU! Runtime > Change runtime type > T4 GPU, then re-run this cell.")

In [ ]:
import sys
from pathlib import Path

# Make notebooks/nb_utils.py importable no matter where the kernel was started
for candidate in (Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent / "notebooks"):
    if (candidate / "nb_utils.py").exists():
        sys.path.insert(0, str(candidate))
        break

import matplotlib.pyplot as plt
import pandas as pd

import nb_utils

ROOT = nb_utils.bootstrap()   # chdir to the repository root + print the device

In [ ]:
EPOCHS = 25

baseline_config = nb_utils.write_config(
    "exp_baseline.py",
    name="baseline",
    model="CRNTiny()",
    docstring="Baseline: the stock nanoSE configuration, unchanged.",
    epochs=EPOCHS,
)
print(baseline_config.read_text())

> **Resuming after a disconnect.** Every finished run is recorded in
> `notebooks/experiment_runs.json`, and the sweep loops skip anything already recorded.
> Re-running the cell after a Colab timeout continues where it stopped instead of
> starting over.

In [ ]:
baseline_run = nb_utils.recall("baseline")
if baseline_run is None:
    baseline_run = nb_utils.remember("baseline", nb_utils.run_training(baseline_config))
print("baseline run:", baseline_run)

## Results

Epoch 0 is the validation pass *before any training*, i.e. the metrics of the
unprocessed noisy input. It is the number every improvement should be measured
against.

In [ ]:
history = nb_utils.load_metrics(baseline_run)
history[["epoch", "val_si_sdr", "pesq", "estoi", "dnsmos"]].round(3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, metric in zip(axes.ravel(), ["val_si_sdr", "pesq", "estoi", "dnsmos"]):
    nb_utils.plot_curves([baseline_run], labels=["baseline"], metric=metric, ax=ax)
    unprocessed = history.loc[history.epoch == 0, metric]
    if len(unprocessed):
        ax.axhline(float(unprocessed.iloc[0]), color=nb_utils.INK_SOFT, linestyle="--", linewidth=1.2)
        ax.annotate("unprocessed input", (0, float(unprocessed.iloc[0])), xytext=(6, 6),
                    textcoords="offset points", color=nb_utils.INK_SOFT, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
best = history[history.epoch > 0].loc[lambda d: d.val_si_sdr.idxmax()]
start = history[history.epoch == 0].iloc[0]

print(f"unprocessed input : SI-SDR {start.val_si_sdr:6.2f} dB | PESQ {start.pesq:.2f} | "
      f"eSTOI {start.estoi:.2f} | DNSMOS {start.dnsmos:.2f}")
print(f"best epoch ({int(best.epoch):2d})   : SI-SDR {best.val_si_sdr:6.2f} dB | PESQ {best.pesq:.2f} | "
      f"eSTOI {best.estoi:.2f} | DNSMOS {best.dnsmos:.2f}")
print(f"improvement       : {best.val_si_sdr - start.val_si_sdr:+.2f} dB SI-SDR, "
      f"{best.pesq - start.pesq:+.2f} PESQ")
if "cumulative_time_min" in history:
    print(f"wall-clock        : {history.cumulative_time_min.max():.1f} min for {EPOCHS} epochs")

## Observations

*Fill in after the run — a template:*

* SI-SDR rises steeply for the first few epochs and then flattens; the cosine
  schedule with 500 warmup steps means the effective learning rate is still
  ramping during epoch 1.
* PESQ and eSTOI keep improving after SI-SDR has flattened — the magnitude MSE
  term and the SI-SDR term optimise slightly different things.
* The gap between epoch 0 and the best epoch is the headroom every later
  experiment is competing for.

**Next:** `02_activation_study.ipynb` (Task 1).